# Обучение NER-модели для анонимизации персональных данных

**Цель:** Дообучение предобученной модели DeepPavlov/rubert-base-cased для распознавания именованных сущностей (NER) в русскоязычных текстах.

**Задача:** Распознавание 9 типов сущностей:
- `PERSON` — ФИО физических лиц
- `ORG` — названия организаций
- `ADDRESS` — адреса (улицы, дома, квартиры)
- `DOC_ID` — номера документов (паспорт, ИНН и т.д.)
- `MONEY` — денежные суммы
- `PHONE` — телефонные номера
- `DATE` — даты
- `EMAIL` — электронные адреса
- `LOC` — географические локации (города, страны)

**Схема разметки:** BIO (Begin-Inside-Outside)

## 1. Импорт библиотек

In [ ]:
import json
import random
import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    Trainer,
    TrainingArguments,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    pipeline
)

from seqeval.metrics import precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import train_test_split

## 2. Конфигурация и гиперпараметры

In [3]:
# Конфигурация модели
MODEL_NAME = "DeepPavlov/rubert-base-cased"

# Гиперпараметры обучения
BATCH_SIZE = 8           # Размер батча
EPOCHS = 50              # Максимальное количество эпох
LR = 3e-5                # Learning rate
EARLY_STOPPING = 3       # Patience для раннего останова

# Параметры чанкинга
MAX_TOKENS = 400         # Максимальная длина чанка (в токенах)
STRIDE = 125             # Перекрытие между чанками (~31%)
RANDOM_STATE = 42

# Типы сущностей для распознавания
LABELS = ["PERSON", "ORG", "ADDRESS", "DOC_ID", "MONEY", "PHONE", "DATE", "EMAIL", "LOC"]

In [4]:
# Фиксация seed для воспроизводимости результатов
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# Детерминированное поведение CUDA
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Определение устройства
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {device}")

Устройство: cuda


In [5]:
# Формирование списка меток в формате BIO
# O — вне сущности
# B-XXX — начало сущности типа XXX
# I-XXX — продолжение сущности типа XXX

label_list = ["O"]
for lbl in LABELS:
    label_list.append(f"B-{lbl}")
    label_list.append(f"I-{lbl}")

# Словари для преобразования меток
label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}

print(f"Всего меток: {len(label_list)}")
print(f"Метки: {label_list}")

Всего меток: 19
Метки: ['O', 'B-PERSON', 'I-PERSON', 'B-ORG', 'I-ORG', 'B-ADDRESS', 'I-ADDRESS', 'B-DOC_ID', 'I-DOC_ID', 'B-MONEY', 'I-MONEY', 'B-PHONE', 'I-PHONE', 'B-DATE', 'I-DATE', 'B-EMAIL', 'I-EMAIL', 'B-LOC', 'I-LOC']


## 3. Загрузка и подготовка данных

Данные размечены в Label Studio в формате JSON. Каждый документ содержит текст и список span-аннотаций с координатами начала/конца и типом сущности.

In [ ]:
def load_labelstudio_json(path):
    """
    Загружает аннотации из формата Label Studio JSON.
    
    Args:
        path: Путь к JSON-файлу с разметкой
        
    Returns:
        Список кортежей (текст, список_span_аннотаций)
    """
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    documents = []

    for item in data:
        text = item["data"]["text"]
        result = item["annotations"][0]["result"]

        spans = []
        for r in result:
            spans.append({
                "start": r["value"]["start"],
                "end": r["value"]["end"],
                "label": r["value"]["labels"][0]
            })

        documents.append((text, spans))

    return documents

In [ ]:
def tokenize_and_align_labels(text, spans):
    """
    Токенизирует текст и выравнивает метки по токенам.
    
    Использует offset_mapping для корректного сопоставления
    символьных позиций span-аннотаций с позициями токенов.
    
    Args:
        text: Исходный текст документа
        spans: Список аннотаций с полями start, end, label
        
    Returns:
        Словарь с input_ids и labels
    """
    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        return_special_tokens_mask=True,
        truncation=False
    )

    # Инициализация меток: -100 для спец-токенов, "O" для остальных
    labels = []
    for is_special in encoding["special_tokens_mask"]:
        labels.append(-100 if is_special else "O")

    # Присвоение BIO-меток на основе span-аннотаций
    for span in spans:
        first_token = True
        for i, (token_start, token_end) in enumerate(encoding["offset_mapping"]):
            if labels[i] == -100:
                continue

            # Проверка пересечения токена со span
            if token_end <= span["start"] or token_start >= span["end"]:
                continue

            # B- для первого токена сущности, I- для остальных
            if first_token:
                labels[i] = f"B-{span['label']}"
                first_token = False
            else:
                labels[i] = f"I-{span['label']}"

    # Удаляем служебные поля
    encoding.pop("offset_mapping")
    encoding.pop("special_tokens_mask")
    encoding["labels"] = labels
    
    return encoding

In [ ]:
def fix_bio(labels):
    """
    Исправляет некорректные BIO-последовательности после чанкинга.
    
    Проблема: при разбиении на чанки I-XXX может оказаться
    в начале чанка без предшествующего B-XXX.
    
    Решение: заменяем такие I-XXX на B-XXX.
    
    Args:
        labels: Список меток (строки или -100 для спец-токенов)
        
    Returns:
        Исправленный список меток
    """
    fixed_labels = []
    prev_label = "O"

    for label in labels:
        # Спец-токены пропускаем
        if label == -100:
            fixed_labels.append(-100)
            prev_label = "O"
            continue

        # I-XXX без предшествующего B-XXX -> B-XXX
        if label.startswith("I-"):
            entity_type = label[2:]
            if prev_label == "O" or prev_label[2:] != entity_type:
                label = f"B-{entity_type}"

        fixed_labels.append(label)
        prev_label = label

    return fixed_labels


def chunk_encoding(encoding):
    """
    Разбивает длинную последовательность на чанки с перекрытием.
    
    Параметры MAX_TOKENS и STRIDE определяют размер чанка
    и величину перекрытия между соседними чанками.
    
    Пустые чанки (только O-метки) пропускаются для экономии памяти.
    
    Args:
        encoding: Словарь с input_ids и labels
        
    Returns:
        Список чанков, каждый — словарь с input_ids, attention_mask, labels
    """
    chunks = []
    input_ids = encoding["input_ids"]
    labels = encoding["labels"]
    
    n = len(input_ids)
    start = 0

    while start < n:
        end = min(start + MAX_TOKENS, n)

        chunk_input_ids = input_ids[start:end]
        chunk_labels = labels[start:end]

        # Исправляем BIO-метки после разрезания
        chunk_labels = fix_bio(chunk_labels)

        # Преобразуем строковые метки в числовые id
        chunk_label_ids = [
            label if label == -100 else label2id[label]
            for label in chunk_labels
        ]

        # Пропускаем пустые чанки (без сущностей)
        if all(l in (-100, label2id["O"]) for l in chunk_label_ids):
            if end == n:
                break
            start = end - STRIDE
            continue

        chunks.append({
            "input_ids": chunk_input_ids,
            "attention_mask": [1] * len(chunk_input_ids),
            "labels": chunk_label_ids
        })

        if end == n:
            break

        start = end - STRIDE

    return chunks

In [ ]:
def prepare_split(docs):
    """
    Подготавливает выборку: токенизация + чанкинг.
    
    Args:
        docs: Список документов (текст, spans)
        
    Returns:
        Список всех чанков из всех документов
    """
    all_chunks = []
    for text, spans in docs:
        encoding = tokenize_and_align_labels(text, spans)
        chunks = chunk_encoding(encoding)
        all_chunks.extend(chunks)
    return all_chunks

In [12]:
# Инициализация токенизатора
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Токенизатор: {MODEL_NAME}")

Токенизатор: DeepPavlov/rubert-base-cased


In [ ]:
# Загрузка аннотированных данных
DATA_PATH = "dataset.json"
documents = load_labelstudio_json(DATA_PATH)
print(f"Загружено документов: {len(documents)}")

# Разделение на train/val/test (70/15/15)
train_docs, temp_docs = train_test_split(
    documents, 
    test_size=0.3, 
    random_state=RANDOM_STATE
)

val_docs, test_docs = train_test_split(
    temp_docs, 
    test_size=0.5, 
    random_state=RANDOM_STATE
)

Загружено документов: 502


In [14]:
# Токенизация и чанкинг для каждой выборки
train_data = prepare_split(train_docs)
val_data = prepare_split(val_docs)
test_data = prepare_split(test_docs)

In [21]:
# Статистика по выборкам
print(f"\nДокументы:")
print(f"Train: {len(train_docs)}")
print(f"Val: {len(val_docs)}")
print(f"Test: {len(test_docs)}")

print(f"\nЧанки:")
print(f"Train: {len(train_data)}")
print(f"Val: {len(val_data)}")
print(f"Test: {len(test_data)}")


Документы:
Train: 351
Val: 75
Test: 76

Чанки:
Train: 393
Val: 98
Test: 90


In [22]:
# Сохранение подготовленных данных для обучения
with open("data/train_v2.json", "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False)

with open("data/val_v2.json", "w", encoding="utf-8") as f:
    json.dump(val_data, f, ensure_ascii=False)

with open("data/test_v2.json", "w", encoding="utf-8") as f:
    json.dump(test_data, f, ensure_ascii=False)

## 4. Обучение модели

Используем `Trainer` API из библиотеки Transformers для fine-tuning модели. 

**Основные настройки:**
- Mixed precision (fp16) для ускорения обучения
- Gradient accumulation для эффективного использования памяти
- Early stopping для предотвращения переобучения
- Выбор лучшей модели по F1-score на валидации

In [23]:
def compute_metrics(eval_pred):
    """
    Вычисляет метрики качества NER на основе seqeval.
    
    Метрики:
        - Precision: точность (доля правильных среди предсказанных)
        - Recall: полнота (доля найденных среди истинных)
        - F1: гармоническое среднее precision и recall
        
    Также возвращает детальный отчёт по каждому классу сущностей.
    """
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    # Преобразуем числовые id в строковые метки
    true_labels = []
    pred_labels = []

    for pred, lab in zip(predictions, labels):
        curr_true = []
        curr_pred = []

        for p_i, l_i in zip(pred, lab):
            # Пропускаем спец-токены
            if l_i == -100:
                continue
            curr_true.append(id2label[l_i])
            curr_pred.append(id2label[p_i])

        true_labels.append(curr_true)
        pred_labels.append(curr_pred)

    # Основные метрики
    metrics = {
        "precision": precision_score(true_labels, pred_labels),
        "recall": recall_score(true_labels, pred_labels),
        "f1": f1_score(true_labels, pred_labels),
    }

    # Детальный отчёт по классам
    metrics["per_class"] = classification_report(
        true_labels, pred_labels, output_dict=True
    )

    return metrics

In [24]:
class NERDataset(Dataset):
    """
    PyTorch Dataset для загрузки подготовленных чанков.
    
    Ожидает JSON-файл со списком словарей, каждый содержит:
        - input_ids: список токенов
        - attention_mask: маска внимания (опционально)
        - labels: числовые метки
    """
    
    def __init__(self, path: str):
        with open(path, "r", encoding="utf-8") as f:
            self.data = json.load(f)

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> dict:
        item = self.data[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(item.get("attention_mask", [1] * len(item["input_ids"])), dtype=torch.long),
            "labels": torch.tensor(item["labels"], dtype=torch.long)
        }

In [25]:
# Создание датасетов
train_dataset = NERDataset("data/train_v2.json")
val_dataset = NERDataset("data/val_v2.json")
test_dataset = NERDataset("data/test_v2.json")

In [26]:
# Инициализация модели для классификации токенов
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True  # Игнорируем несовпадение размеров head
)

print(f"Модель: {MODEL_NAME}")
print(f"Количество меток: {len(label_list)}")

Some weights of BertForTokenClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Модель: DeepPavlov/rubert-base-cased
Количество меток: 19


In [27]:
# Настройки обучения
training_args = TrainingArguments(
    output_dir="./ner_model",

    eval_strategy="epoch", 
    save_strategy="epoch", 
    load_best_model_at_end=True, 
    metric_for_best_model="f1", 
    greater_is_better=True, 
    
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01, 
    
    fp16=True, 
    gradient_accumulation_steps=4, 
    max_grad_norm=1.0, 
    
    logging_steps=50,
    report_to="none"
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [28]:
# Data collator для динамического паддинга батчей
data_collator = DataCollatorForTokenClassification(tokenizer)

In [29]:
# Инициализация Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING)
    ]
)

In [30]:
# Запуск обучения
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Per Class
0,No log,1.044315,0.209259,0.103291,0.138311,"{'ADDRESS': {'precision': 0.5084745762711864, 'recall': 0.5084745762711864, 'f1-score': 0.5084745762711864, 'support': 59}, 'DATE': {'precision': 0.1388888888888889, 'recall': 0.1111111111111111, 'f1-score': 0.12345679012345678, 'support': 180}, 'DOC_ID': {'precision': 0.1694915254237288, 'recall': 0.16260162601626016, 'f1-score': 0.16597510373443983, 'support': 123}, 'EMAIL': {'precision': 0.07142857142857142, 'recall': 0.0967741935483871, 'f1-score': 0.0821917808219178, 'support': 31}, 'LOC': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 112}, 'MONEY': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 69}, 'ORG': {'precision': 0.4146341463414634, 'recall': 0.17525773195876287, 'f1-score': 0.24637681159420288, 'support': 97}, 'PERSON': {'precision': 0.06818181818181818, 'recall': 0.00847457627118644, 'f1-score': 0.015075376884422108, 'support': 354}, 'PHONE': {'precision': 0.21739130434782608, 'recall': 0.2898550724637681, 'f1-score': 0.24844720496894407, 'support': 69}, 'micro avg': {'precision': 0.20925925925925926, 'recall': 0.10329067641681901, 'f1-score': 0.1383108935128519, 'support': 1094}, 'macro avg': {'precision': 0.1764989812092759, 'recall': 0.15028320973785136, 'f1-score': 0.15444418271095223, 'support': 1094}, 'weighted avg': {'precision': 0.1438917908344515, 'recall': 0.10329067641681901, 'f1-score': 0.11111809549508046, 'support': 1094}}"
1,No log,0.392675,0.745956,0.716636,0.731002,"{'ADDRESS': {'precision': 0.7605633802816901, 'recall': 0.9152542372881356, 'f1-score': 0.8307692307692307, 'support': 59}, 'DATE': {'precision': 0.9106145251396648, 'recall': 0.9055555555555556, 'f1-score': 0.9080779944289693, 'support': 180}, 'DOC_ID': {'precision': 0.5953757225433526, 'recall': 0.8373983739837398, 'f1-score': 0.6959459459459459, 'support': 123}, 'EMAIL': {'precision': 0.8285714285714286, 'recall': 0.9354838709677419, 'f1-score': 0.8787878787878788, 'support': 31}, 'LOC': {'precision': 0.6363636363636364, 'recall': 0.0625, 'f1-score': 0.11382113821138211, 'support': 112}, 'MONEY': {'precision': 0.48717948717948717, 'recall': 0.2753623188405797, 'f1-score': 0.35185185185185186, 'support': 69}, 'ORG': {'precision': 0.4700854700854701, 'recall': 0.5670103092783505, 'f1-score': 0.5140186915887851, 'support': 97}, 'PERSON': {'precision': 0.9104046242774566, 'recall': 0.8898305084745762, 'f1-score': 0.9, 'support': 354}, 'PHONE': {'precision': 0.4875, 'recall': 0.5652173913043478, 'f1-score': 0.5234899328859061, 'support': 69}, 'micro avg': {'precision': 0.7459562321598477, 'recall': 0.716636197440585, 'f1-score': 0.7310023310023309, 'support': 1094}, 'macro avg': {'precision': 0.6762953638269096, 'recall': 0.6615125072992253, 'f1-score': 0.6351958516077721, 'support': 1094}, 'weighted avg': {'precision': 0.7441571495438101, 'recall': 0.716636197440585, 'f1-score': 0.7010233664689826, 'support': 1094}}"
2,No log,0.220124,0.638479,0.752285,0.690726,"{'ADDRESS': {'precision': 0.7605633802816901, 'recall': 0.9152542372881356, 'f1-score': 0.8307692307692307, 'support': 59}, 'DATE': {'precision': 0.9777777777777777, 'recall': 0.9777777777777777, 'f1-score': 0.9777777777777777, 'support': 180}, 'DOC_ID': {'precision': 0.689873417721519, 'recall': 0.8861788617886179, 'f1-score': 0.7758007117437722, 'support': 123}, 'EMAIL': {'precision': 0.8529411764705882, 'recall': 0.9354838709677419, 'f1-score': 0.8923076923076922, 'support': 31}, 'LOC': {'precision': 0.6904761904761905, 'recall': 0.7767857142857143, 'f1-score': 0.7310924369747899, 'support': 112}, 'MONEY': {'precision': 0.765625, 'recall': 0.7101449275362319, 'f1-score': 0.736842105263158, 'support': 69}, 'ORG': {'precision': 0.6476190476190476, 'recall': 0.7010309278350515, 'f1-score': 0.6732673267326732, 'support': 97}, 'PERSON': {'precision': 0.40408163265306124, 'recall': 0.559322033898305, 'f1-score': 0.46919431279620855, 'suppo

/notebooks/myenv/lib/python3.9/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


TrainOutput(global_step=144, training_loss=0.289619031879637, metrics={'train_runtime': 212.4869, 'train_samples_per_second': 92.476, 'train_steps_per_second': 2.824, 'total_flos': 766876849355712.0, 'train_loss': 0.289619031879637, 'epoch': 11.96})

In [31]:
# Очистка памяти GPU
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [32]:
# Сохранение лучшей модели и токенизатора
MODEL_OUTPUT_DIR = "ner_model_best"
trainer.save_model(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

('ner_model_best/tokenizer_config.json',
 'ner_model_best/special_tokens_map.json',
 'ner_model_best/vocab.txt',
 'ner_model_best/added_tokens.json',
 'ner_model_best/tokenizer.json')

## 5. Оценка качества модели

Оцениваем модель на тестовой выборке и анализируем результаты по каждому классу сущностей.

In [33]:
# Оценка на тестовой выборке
test_metrics = trainer.evaluate(test_dataset)

print("РЕЗУЛЬТАТЫ НА ТЕСТОВОЙ ВЫБОРКЕ")
print(f"Precision: {test_metrics['eval_precision']:.4f}")
print(f"Recall: {test_metrics['eval_recall']:.4f}")
print(f"F1-score: {test_metrics['eval_f1']:.4f}")

РЕЗУЛЬТАТЫ НА ТЕСТОВОЙ ВЫБОРКЕ
Precision: 0.9579
Recall: 0.9616
F1-score: 0.9598


In [34]:
# Детальные метрики по каждому классу сущностей
per_class_df = pd.DataFrame(test_metrics['eval_per_class']).T
per_class_df = per_class_df.round(4)

entity_classes = [col for col in per_class_df.index if col not in ['micro avg', 'macro avg', 'weighted avg']]
per_class_df.loc[entity_classes]

,precision,recall,f1-score,support
ADDRESS,0.8987,0.9103,0.9045,78.0
DATE,0.9728,0.9728,0.9728,184.0
DOC_ID,0.9759,0.9701,0.9730,167.0
EMAIL,0.9796,0.9231,0.9505,52.0
LOC,0.9574,0.9783,0.9677,138.0
MONEY,0.9839,1.0000,0.9919,61.0
ORG,0.8814,0.9123,0.8966,114.0
PERSON,0.9750,0.9750,0.9750,160.0
PHONE,0.9885,0.9773,0.9829,88.0


## 6. Инференс
Демонстрация работы обученной модели на примерах текстов.

In [40]:
# Создание pipeline для инференса
ner_pipeline = pipeline(
    task="token-classification",
    model=MODEL_OUTPUT_DIR,
    tokenizer=MODEL_OUTPUT_DIR,
    aggregation_strategy="simple"  # Объединяет токены в сущности
)

# Пример использования
example_text = "Петр Иванов живет в Воронеже и работает в Газпроме."
entities = ner_pipeline(example_text)

print("Найденные сущности:")
for ent in entities:
    print(f"  {ent['entity_group']:>10}: {ent['word']!r} (score: {ent['score']:.3f})")

Device set to use cuda:0
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Найденные сущности:
      PERSON: 'Петр Иванов' (score: 0.867)
         LOC: 'Воронеже' (score: 0.835)
         ORG: 'Газпроме' (score: 0.383)
